# **Apple**

### **Rate Limiter**

#### **Token Bucket**

In [ ]:
import time
import threading

In [8]:
import time
import threading

class tokenBucket:
  """
  Token Bucket Rate Limiter:
    - rate_per_sec: tokens added per second
    - capacity: max token that the bucket can hold
    - allow(cost): spend cost if tokens are available. 1 token default
  """

  def __init__(self, rate_per_sec: float, capacity: float):
    if rate_per_sec <= 0:
      raise ValueError("rate_per_sec should be > 0")
    if capacity <= 0:
      raise ValueError("capacity should be > 0")

    self.rate = rate_per_sec
    self.cap = capacity
    self.tokens = capacity
    self.last_time = time.monotonic()
    self._lock = threading.Lock()

  def refill(self, now):
    elapsed = now - self.last_time
    if elapsed <= 0:
      return

    self.tokens = min(self.cap, self.tokens + elapsed * self.rate)
    self.last_time = now

  def allow(self, cost: float = 1.0) -> bool:
    if cost <= 0:
      return True # not spending anything is always allowed

    now = time.monotonic()
    with self._lock:
      self.refill(now)
      if self.tokens >= cost:
        self.tokens -= 1
        return True, self.tokens
      return False, self.tokens

In [9]:
def test_bucket():
  bucket = tokenBucket(rate_per_sec=2, capacity=5)
  for i in range(10):
    result = bucket.allow(1.0)
    print(f"request {i + 1}: {'ALLOWED' if result[0] else 'REJECTED'} (Tolens left: {bucket.tokens:.2f})")
    time.sleep(0.2)


  '''for i in range(3):
    result = bucket.allow(1.0)
    print(f"request {i + 1}: {'ALLOWED' if result else 'REJECTED'} (Tolens left: {bucket.tokens:.2f})")'''

if __name__ == "__main__":
  test_bucket()

request 1: ALLOWED (Tolens left: 4.00)
request 2: ALLOWED (Tolens left: 3.40)
request 3: ALLOWED (Tolens left: 2.80)
request 4: ALLOWED (Tolens left: 2.20)
request 5: ALLOWED (Tolens left: 1.60)
request 6: ALLOWED (Tolens left: 1.00)
request 7: ALLOWED (Tolens left: 0.40)
request 8: REJECTED (Tolens left: 0.80)
request 9: ALLOWED (Tolens left: 0.20)
request 10: REJECTED (Tolens left: 0.60)


#### **Token bucket with variable cost**

In [10]:

VIDEO_4K = 25
VIDEO_1080 = 10
VIDEO_720 = 5

bucket = tokenBucket(rate_per_sec=20, capacity=50)

def process_stream(quality_type, tag):
    # Unpack both the boolean and the float
    allowed, remaining = bucket.allow(cost=quality_type)

    if allowed:
        print(f"[OK]... Transcoding {tag} (Cost: {quality_type})")
        print(f"Remaining tokens: {remaining:.2f}")
    else:
        print(f"[DENIED]... Capacity exceeded for {tag}. Dropping frame/Lowering bitrate")
        print(f"Remaining tokens: {remaining:.2f}")

# Simulation
process_stream(VIDEO_4K, "VIDEO_4K")
process_stream(VIDEO_4K, "VIDEO_4K")

print(f"\n--- sleep 1 sec (refilling 20 tokens) ---\n")
time.sleep(1)

process_stream(VIDEO_720, "VIDEO_720")
process_stream(VIDEO_4K, "VIDEO_4K")

[OK]... Transcoding VIDEO_4K (Cost: 25)
Remaining tokens: 49.00
[OK]... Transcoding VIDEO_4K (Cost: 25)
Remaining tokens: 48.00

--- sleep 1 sec (refilling 20 tokens) ---

[OK]... Transcoding VIDEO_720 (Cost: 5)
Remaining tokens: 49.00
[OK]... Transcoding VIDEO_4K (Cost: 25)
Remaining tokens: 48.01


#### **Leaky bucket**

## 1. The Leaky Bucket (Traffic Shaping)

**Interviewer:** *"You explained the Token Bucket allows for bursts. What if I need a system where the output rate is strictly constant, regardless of how bursty the input is? For example, a system feeding a fixed-bandwidth network link."*

### The Response Flow

**The Concept:**
"For that, I would use the **Leaky Bucket** algorithm. Think of it as a physical bucket with a small hole at the bottom. You can pour water in (packets) as fast as you want, but the water leaks out (is processed) at a **constant, predictable rate**. If the bucket overflows, the packets are dropped."

**The Implementation (Pseudo-code Logic):**
"In Python, instead of tracking 'tokens' that represent budget, we track a 'queue size' that represents the current volume of the bucket."

In [ ]:
import time

class leakyBucket:
  """
  Leaky Bucket:
    - leak_rate: process a constant rate/amount of packets/requests
    - capacity: maximum number of packets/requests the bucket can hold
    - allow(packet_size): allow teh packet only if the space/memory is available
    - Monotonic time to prevent wall clock jumps
  """
  def __init__(self, capacity, leak_rate):
    if leak_rate <= 0:
      raise ValueError("leak_rate should be > 0")
    if capacity <= 0:
      raise ValueError("capacity should be > 0")
    self.cap = capacity
    self.leak_rate = leak_rate
    self.current_volume = 0 # to track the queue size
    self.last_check_time = time.monotonic()

  # Packets leaked
  def leak(self, now):
    elapsed = now - self.last_check_time

    if elapsed <= 0:
      return

    leaked_amount = elapsed * self.leak_rate
    self.current_volume = max(0, self.current_volume - leaked_amount)
    self.last_check_time = now

  def allow(self, packet_size: float = 1.0) -> bool:
    if packet_size <= 0:
      return True # empty stream can always be processed

    now = time.monotonic()
    self.leak(now)
    if self.current_volume + packet_size <= self.cap:
      self.current_volume += packet_size
      return True, self.current_volume
    return False, self.current_volume

In [ ]:
def test_leaky_bucket():
  lb = leakyBucket(capacity=5, leak_rate=2)

  print(f"capcity: {lb.cap}")
  for i in range(7):
    allowed, volume = lb.allow(1.0)
    status = "ACCEPTED" if allowed else "DROPPED (overflow)"
    print(f"Packet {i+1}: {status} | Current Volume: {volume:.2f}")

  print(f"sleep for 1.5 sec. leaks ~3 tokens")
  time.sleep(1.5)

  print(f"CUrrent Volume: {max(0, lb.current_volume - (time.monotonic() - lb.last_check_time) * lb.leak_rate):.2f}")

  for i in range(5):
    allowed, volume = lb.allow(1.0)
    status = "ACCEPTED" if allowed else "DROPPED (overflow)"
    print(f"Packet {i+8}: {status} | Current Volume: {volume:.2f}")

if __name__ == "__main__":
  test_leaky_bucket()

capcity: 5
Packet 1: ACCEPTED | Current Volume: 1.00
Packet 2: ACCEPTED | Current Volume: 2.00
Packet 3: ACCEPTED | Current Volume: 3.00
Packet 4: ACCEPTED | Current Volume: 4.00
Packet 5: ACCEPTED | Current Volume: 5.00
Packet 6: DROPPED (overflow) | Current Volume: 5.00
Packet 7: DROPPED (overflow) | Current Volume: 5.00
sleep for 1.5 sec. leaks ~3 tokens
CUrrent Volume: 2.00
Packet 8: ACCEPTED | Current Volume: 3.00
Packet 9: ACCEPTED | Current Volume: 4.00
Packet 10: ACCEPTED | Current Volume: 5.00
Packet 11: DROPPED (overflow) | Current Volume: 5.00
Packet 12: DROPPED (overflow) | Current Volume: 5.00


To round out your toolkit, we need to cover the **Fixed Window Counter** and the **Sliding Window Counter**.

In a System Design interview, the Fixed Window is usually the "starting point" (the naive solution), while the Sliding Window is the "optimization" that fixes the boundary issues.

---

## 1. Fixed Window Counter

This is the simplest form of rate limiting. It divides time into discrete blocks (e.g., 60-second windows).

### The Implementation

In [23]:
import time
import threading

class FixedWindowCounter:
    def __init__(self, limit: int, window_size: int):
        self.limit = limit
        self.window_size = window_size # in seconds
        self.counter = 0
        self.last_window = int(time.time() / window_size)
        self._lock = threading.Lock()

    def allow(self) -> bool:
        with self._lock:
            current_window = int(time.time() / self.window_size)

            # If the window has changed, reset the counter
            if current_window > self.last_window:
                self.last_window = current_window
                self.counter = 0

            if self.counter < self.limit:
                self.counter += 1
                return True
            return False

In [32]:
from typing import Counter
import time

def sleep_until_next_window(window_size: int):
    now = time.time()
    next_boundary = (int(now / window_size) + 1) * window_size
    time.sleep(max(0, next_boundary - now) + 0.01)  # +epsilon to be safely inside next window

def test_fixed_window():
  limiter = fixedWindiwCounter(5, 10)
  print(f"Limit: {limiter.limit} | Window Size: {limiter.window_size}")
  print("")
  for i in range(10):
    status = "ALLOWED" if limiter.allow() else "DENIED"
    print(f"Packet: {i+1} | Status: {status} | Counter: {limiter.counter}")

  print("")
  print(f"sleeping")
  print("")
  time.sleep(3)

  for i in range(10):
    status = "ALLOWED" if limiter.allow() else "DENIED"
    print(f"Packet: {i+11} | Status: {status} | Counter: {limiter.counter}")

if __name__ == "__main__":
  test_fixed_window()

Limit: 5 | Window Size: 10

Packet: 1 | Status: ALLOWED | Counter: 1
Packet: 2 | Status: ALLOWED | Counter: 2
Packet: 3 | Status: ALLOWED | Counter: 3
Packet: 4 | Status: ALLOWED | Counter: 4
Packet: 5 | Status: ALLOWED | Counter: 5
Packet: 6 | Status: DENIED | Counter: 5
Packet: 7 | Status: DENIED | Counter: 5
Packet: 8 | Status: DENIED | Counter: 5
Packet: 9 | Status: DENIED | Counter: 5
Packet: 10 | Status: DENIED | Counter: 5

sleeping

Packet: 11 | Status: ALLOWED | Counter: 1
Packet: 12 | Status: ALLOWED | Counter: 2
Packet: 13 | Status: ALLOWED | Counter: 3
Packet: 14 | Status: ALLOWED | Counter: 4
Packet: 15 | Status: ALLOWED | Counter: 5
Packet: 16 | Status: DENIED | Counter: 5
Packet: 17 | Status: DENIED | Counter: 5
Packet: 18 | Status: DENIED | Counter: 5
Packet: 19 | Status: DENIED | Counter: 5
Packet: 20 | Status: DENIED | Counter: 5


### The "Interview Call-out": The Boundary Problem

"The main flaw with Fixed Window is that a user can double their rate at the edges. If the limit is 100/min, a user can send 100 requests at 11:59:59 and another 100 at 12:00:01. In two seconds, they sent 200 requests, potentially crashing the server."

---

## 2. Sliding Window Counter (Weighted Average)

This is the "Senior Engineer" solution. It provides the accuracy of a log-based system but uses the $O(1)$ memory of a counter-based system.

### How it Works

It calculates a weighted average of the **previous window** and the **current window**.
If we are 25% into the current minute, we calculate:


$$\text{Count} = (\text{Previous Count} \times 75\%) + \text{Current Count}$$

### The Implementation

In [45]:
class SlidingWindowCounter:
  """
  Sliding Window Counter:
    - limit: max requests to be processed
    - window_size: time in seconds
    - prev_count: requests processed in the previous window
    - curr_count: requests processed in the current window
  """
  def __init__(self, limit: int, window_size: int = 60):
    self.limit = limit
    self.window_size = window_size
    self.prev_count = 0
    self.curr_count = 0
    self.current_window = int(time.time() / window_size)
    self._lock = threading.Lock()

  def allow(self) -> bool:
    with self._lock:
      now = time.time()
      now_window = int(now / self.window_size)

      # If we've moved to a new window, shift the counts
      if now_window > self.current_window:
        # If it's a brand new window, the old 'current' becomes 'previous'
        if now_window == self.current_window + 1:
          self.prev_count = self.curr_count
        else:
          self.prev_count = 0
          self.curr_count = 0
          self.current_window = now_window

        # Calculate the weighted average
        # (time_elapsed_in_current_window / window_size)
        elapsed_percentage = (now % self.window_size) / self.window_size
        weighted_count = self.prev_count * (1 - elapsed_percentage) + self.curr_count

        if weighted_count < self.limit:
          self.curr_count += 1
          return True
        return False

In [46]:
import time

def test_sliding_window_counter():
    limit = 5
    window = 10
    limiter = SlidingWindowCounter(limit, window)

    def snapshot():
        now = time.time()
        now_window = int(now / window)
        elapsed_pct = (now % window) / window
        weighted = limiter.prev_count * (1 - elapsed_pct) + limiter.curr_count
        return now, now_window, elapsed_pct, weighted

    print(f"--- Testing Sliding Window Counter (Limit: {limit}/{window}s) ---")

    # 1) Align near the end of the current window (to create the classic boundary scenario)
    print("\n[Step A] Waiting until we're near the end of Window A...")
    while True:
        now = time.time()
        frac = (now % window) / window
        if frac >= 0.85:   # last 15% of the window
            break
        time.sleep(0.01)

    # 2) Burst requests in Window A (you'll hit the limit)
    print("[Step B] Sending 7 requests near the end of Window A:")
    for i in range(7):
        t, w, e, weighted_before = snapshot()
        allowed = limiter.allow()
        _, _, _, weighted_after = snapshot()
        print(
            f"Req {i+1:02d} | win={w} t%={e:.3f} "
            f"prev={limiter.prev_count} curr={limiter.curr_count - (1 if allowed else 0)} "
            f"weighted_before={weighted_before:.3f} -> {'Allowed' if allowed else 'Denied'} "
            f"(weighted_after={weighted_after:.3f}, curr_now={limiter.curr_count})"
        )
        time.sleep(0.01)

    # 3) Wait exactly until the next window boundary
    print("\n[Step C] Waiting for the window boundary...")
    now = time.time()
    next_boundary = (int(now / window) + 1) * window
    time.sleep(max(0, next_boundary - now) + 0.02)  # epsilon so we land inside the new window

    # 4) Immediately burst again in Window B
    print("[Step D] Sending 7 requests right after entering Window B:")
    for i in range(7):
        t, w, e, weighted_before = snapshot()
        allowed = limiter.allow()
        _, _, _, weighted_after = snapshot()
        print(
            f"Req {i+8:02d} | win={w} t%={e:.3f} "
            f"prev={limiter.prev_count} curr={limiter.curr_count - (1 if allowed else 0)} "
            f"weighted_before={weighted_before:.3f} -> {'Allowed' if allowed else 'Denied'} "
            f"(weighted_after={weighted_after:.3f}, curr_now={limiter.curr_count})"
        )
        time.sleep(0.01)

if __name__ == "__main__":
  test_sliding_window_counter()

--- Testing Sliding Window Counter (Limit: 5/10s) ---

[Step A] Waiting until we're near the end of Window A...
[Step B] Sending 7 requests near the end of Window A:
Req 01 | win=177196228 t%=0.851 prev=0 curr=0 weighted_before=0.000 -> Denied (weighted_after=0.000, curr_now=0)
Req 02 | win=177196228 t%=0.852 prev=0 curr=0 weighted_before=0.000 -> Denied (weighted_after=0.000, curr_now=0)
Req 03 | win=177196228 t%=0.853 prev=0 curr=0 weighted_before=0.000 -> Denied (weighted_after=0.000, curr_now=0)
Req 04 | win=177196228 t%=0.854 prev=0 curr=0 weighted_before=0.000 -> Denied (weighted_after=0.000, curr_now=0)
Req 05 | win=177196228 t%=0.855 prev=0 curr=0 weighted_before=0.000 -> Denied (weighted_after=0.000, curr_now=0)
Req 06 | win=177196228 t%=0.856 prev=0 curr=0 weighted_before=0.000 -> Denied (weighted_after=0.000, curr_now=0)
Req 07 | win=177196228 t%=0.857 prev=0 curr=0 weighted_before=0.000 -> Denied (weighted_after=0.000, curr_now=0)

[Step C] Waiting for the window boundary..

---

## Comparison for your Mock Interview

| Feature | Fixed Window | Sliding Window Counter |
| --- | --- | --- |
| **Memory** | $O(1)$ (Very Low) | $O(1)$ (Low) |
| **Accuracy** | Low (Boundary issues) | High (Smooths out edges) |
| **Complexity** | Simple | Moderate |
| **Scalability** | Easy to implement in Redis | Requires slightly more complex Lua logic |

### Summary of the "Big Four" Algorithms:

1. **Token Bucket:** Best for handling bursts.
2. **Leaky Bucket:** Best for smoothing traffic to a constant rate.
3. **Fixed Window:** Good for simple, non-critical limits.
4. **Sliding Window Counter:** Best for high-accuracy limits without $O(N)$ memory cost.

### **Tiny URL Generator**

In a **System Design** interview, the **Key Generation Service (KGS)** is the "Senior Engineer" answer to the TinyURL problem. Most candidates suggest hashing the URL on the fly (e.g., using MD5 or SHA-256), but that leads to **collision handling** and **high latency** issues.

The KGS approach avoids these problems by generating keys *before* they are even needed.

---

## 1. The Core Logic of KGS

Instead of calculating a hash when the user clicks "Shorten," the KGS pre-generates a massive list of random, unique 7-character strings and stores them in a dedicated **Key Database**.

### The Workflow:

1. **Pre-generation:** A background process generates Base62 strings and stores them in a table.
2. **Request:** A user wants to shorten a URL.
3. **Assignment:** The App Server asks the KGS for an unused key.
4. **Mark as Used:** The KGS gives the key to the App Server and marks it as "Used" in the database so it’s never issued again.

---

## 2. Managing Concurrency (The "Locking" Challenge)

In an interview, the interviewer will ask: *"What if two App Servers ask for a key at the same exact time? How do you prevent them from getting the same key?"*

**The "Good" Answer:** Use a database lock.
**The "Senior" Answer:** Use **In-Memory Caching with a Range-Based approach.**

### How to Scale KGS:

* The KGS doesn't read from the DB for every single request. Instead, it loads a **range** of keys (e.g., keys 1,000 to 5,000) into its own memory.
* Even if you have multiple KGS instances, you assign each instance a unique **segment** of keys.
* KGS-Server-A handles keys 1–5,000.
* KGS-Server-B handles keys 5,001–10,000.


* Once an instance runs out of keys in its memory, it goes back to the DB to fetch a new unique segment.

---

## 3. Pseudo-code for the KGS Logic (Python)

In [51]:
import sqlite3 # Using SQLite to simulate a transactional SQL database

class KGS:
    def __init__(self, server_id, segment_size=1000):
        self.server_id = server_id
        self.unused_keys = []
        self.segment_size = segment_size
        self._lock = threading.Lock()
        # In production, this would be a connection string to Postgres/MySQL/DynamoDB
        self.db_conn = sqlite3.connect('keys_metadata.db', check_same_thread=False)

    def _fetch_new_segment_from_db(self):
        """
          Transactional update:
            1. Find the next available range in DB
            2. Mark that range as "Assigned to Server X"
            3. Load those keys into self.unused_keys
        """
        cursor = self.db_conn.cursor()
        try:
            # 1. Start Transaction
            cursor.execute("BEGIN TRANSACTION;")

            # 2. Find the next available range (using an offset or a status flag)
            # We select keys that are NOT yet assigned
            cursor.execute("""
                SELECT key_value FROM key_store
                WHERE status = 'AVAILABLE'
                LIMIT ?;
            """, (self.segment_size,))

            rows = cursor.fetchall()

            if not rows:
                raise Exception("Out of unique keys in the database!")

            # 3. Extract keys and update their status to 'ASSIGNED'
            new_keys = [row[0] for row in rows]

            # Using placeholders to prevent SQL injection and update the specific keys
            cursor.executemany("""
                UPDATE key_store
                SET status = 'ASSIGNED', assigned_to = ?
                WHERE key_value = ?;
            """, [(self.server_id, k) for k in new_keys])

            # 4. Commit Transaction
            self.db_conn.commit()

            # 5. Load into memory
            self.unused_keys = new_keys
            print(f"Server {self.server_id} successfully fetched {len(new_keys)} keys.")

        except Exception as e:
            self.db_conn.rollback()
            print(f"Error fetching segment: {e}")
            raise

---

### Interview Deep Dive: Why this works

#### 1. Atomicity (The "A" in ACID)

By using `BEGIN TRANSACTION` and `COMMIT`, we ensure that the "Select" and "Update" happen as a single, indivisible unit. If the server crashes halfway through, the DB rolls back, and those keys remain `AVAILABLE` for another server.

#### 2. Avoiding Double-Issuance

If you didn't have the transaction, two KGS instances might both run the `SELECT` query at the same time, see the same 1,000 keys, and both try to issue them to users. This would lead to two different long URLs having the same short URL—a catastrophic failure for a URL shortener.

---

### Complexity Analysis

* **Time Complexity:** * `get_key()` from memory: **$O(1)$**.
* `_fetch_new_segment_from_db()`: **$O(K)$** where $K$ is the segment size (Disk I/O). However, since this only happens once every $K$ requests, the **Amortized Time Complexity** per request is still effectively **$O(1)$**.


* **Space Complexity:** **$O(K)$** in memory to store the strings.

---

### How to explain the "Failure Case" in an interview

**Interviewer:** *"What if the KGS server crashes after fetching the keys but before issuing them?"*

**Your Response:** "Since the database has marked those keys as `ASSIGNED` to that specific Server ID, they won't be given to anyone else. When the server restarts, it will simply fetch a *new* range. Yes, those 1,000 keys are 'lost,' but in a Base62 system with 3.5 trillion combinations, losing 1,000 keys is a negligible price to pay for the massive performance gain of not hitting the DB for every single request."

Would you like me to show you how to generate the initial **3.5 trillion keys** in the database efficiently using an incremental counter?

---

## 4. Complexity & Trade-offs

### Time Complexity

* **Reading a Key:** $O(1)$ from memory. This is incredibly fast compared to hashing or database writes.
* **Generating Keys:** $O(N)$ but done **offline**, so it doesn't affect the user's experience.

### Space Complexity

* **Storage:** $O(N)$ where $N$ is the total number of keys. 3.5 trillion keys sounds like a lot, but since each key is only 7 bytes, the total storage is manageable (roughly 25 TB for 3.5 trillion keys).

### Potential Issues to Call Out:

* **Single Point of Failure:** If the KGS goes down, we can't shorten URLs. We need a standby (Replica) KGS.
* **Key Wastage:** If a KGS server crashes, the keys currently in its memory (the segment) are lost. **Interview Answer:** "This is acceptable. We have 3.5 trillion keys; losing a few thousand is a fair trade-off for the massive speed increase of in-memory assignment."

---

## Summary of your Networking/System Design "Big Three"

1. **Socket API & Token Bucket:** Understanding the transport layer and admission control.
2. **Leaky Bucket:** Understanding traffic smoothing and fixed-rate output.
3. **TinyURL & KGS:** Understanding high-scale data mapping, NoSQL, and pre-computation.

Would you like to try a **full mock interview** where I ask you a question and you explain it back using the flow we've built?

### **LRU Cache**

In [ ]:
class Node:
  def __init__(self, key, val):
    self.key, self.val = key, val
    self.prev = self.nxt = None

class LRUCache:
  def __init__(self, capacity):
    self.cap = capacity
    self.cache = {} # HashMap: map key to node

    # left = LRU, right = most recently used
    self.left, self.right = Node(0, 0), Node(0, 0)
    self.left.nxt, self.right.prev = self.right, self.left

  # remove from left
  def remove(self, node):
    prev, nxt = node.prev, node.nxt
    prev.nxt, nxt.prev = nxt, prev

  # insert on the right i.e., in between
  def insert(self, node):
    prev, nxt = self.right.prev, self.right
    prev.nxt = nxt.prev = node
    node.nxt, node.prev = nxt, prev

  def get(self, key):
    if key in self.cache:
      self.remove(self.cache[key])
      self.insert(self.cache[key])
      return self.cache[key].val
    return -1

  def put(self, key, value):
    if key in self.cache:
      self.remove(self.cache[key])
    self.cache[key] = Node(key, value)
    self.insert(self.cache[key])

    if(len(self.cache) > self.cap):
      lru = self.left.nxt
      self.remove(lru)
      del self.cache[lru.key] # remove the lru from the cache

In [ ]:
commands = ["LRUCache", "put", "put", "get", "put", "get", "put", "get", "get", "get"]
params = [[2], [1, 1], [2, 2], [1], [3, 3], [2], [4, 4], [1], [3], [4]]

output = []
obj = None

for i in range(len(commands)):
    cmd = commands[i]
    val = params[i]

    if cmd == "LRUCache":
        obj = LRUCache(val[0])
        output.append(None)
    elif cmd == "put":
        obj.put(val[0], val[1])
        output.append(None)
    elif cmd == "get":
        result = obj.get(val[0])
        output.append(result)

print(output)

[None, None, None, 1, None, -1, None, -1, 3, 4]


### **Two Sum**

In [ ]:
class Solution:
  '''
  nums: list[int]
  target: int
  return: List[int]
  '''
  def twoSum(nums, target):
    prevMap = {} # val : index
    for i, n in enumerate(nums):
      diff = target - n
      if diff in prevMap:
        return [prevMap[diff], i]
      prevMap[n] = i

    return

In [ ]:
twoSum = Solution.twoSum([1, 3, 5, 7], 8)
print(twoSum)

[1, 2]


### **Reverse Integer**

In [ ]:
class Solution:
  def reverseInteger(x: int) -> int:
    '''
    input: int
    return: int
    '''
    res = 0
    if x < 0:
      res = int(str(x)[1:][::-1])
    else:
      res = int(str(x)[::-1])

    if res > 2**31 - 1 or res < -2**31:
      res = 0

    return res

In [ ]:
ri = Solution.reverseInteger(45678)
print(ri)

87654


### **Snapshot Array**
---

### **Problem Description**

Implement a `SnapshotArray` that supports the following interface:

* `SnapshotArray(int length)` initializes an array-like data structure with the given length. **Initially, each element equals 0**.
* `void set(index, val)` sets the element at the given index to be equal to `val`.
* `int snap()` takes a snapshot of the array and returns the `snap_id`: the total number of times we called `snap()` minus 1.
* `int get(index, snap_id)` returns the value at the given index, at the time we took the snapshot with the given `snap_id`.

---

### **Example 1**

**Input:** `["SnapshotArray","set","snap","set","get"]`
`[[3],[0,5],[],[0,6],[0,0]]`

**Output:** `[null,null,0,null,5]`

**Explanation:** ```java
SnapshotArray snapshotArr = new SnapshotArray(3); // set the length to be 3
snapshotArr.set(0,5);  // Set array[0] = 5
snapshotArr.snap();    // Take a snapshot, return snap_id = 0
snapshotArr.set(0,6);
snapshotArr.get(0,0);  // Get the value of array[0] with snap_id = 0, return 5

```
### **Constraints**

* 1 <= length <= 5 * 10^4
* 0 <= index < length
* 0 <= val <= 10^9
* 0 <= snap_id <  (the total number of times we call snap())
* At most (5 * 10^4) calls will be made to set, snap, and get.
```

In [ ]:
class SnapshotArray:
  def __init__(self, length: int):
    self.arr = [[(0, 0)] for _ in range(length)] # initialize elements to (0, 0)
    self.snap_id = 0
    print(f"Array initialized with length {length}. Current snap_id: {self.snap_id}")
    print(f"Current internal state: {self.arr}\n")

  def set(self, index: int, val: int) -> None:
    self.arr[index].append((self.snap_id, val)) # append (snap_id, val) in 'index' of the initialised arr
    print(f"index[{index}] = {val} (recorded during snap_id {self.snap_id})")
    print(f"Current internal state: {self.arr}\n")

  def snap(self) -> int:
    print(f"Closing snap_id {self.snap_id}.")
    self.snap_id += 1
    print(f"Next snap_id will be {self.snap_id}\n")
    return self.snap_id - 1

  def get(self, index: int, snap_id: int) -> int:
    history = self.arr[index]
    print(f"Target: index {index} at snapshot {snap_id}")
    print(f"Timeline to search: {history}")

    left, right = 0, len(history) - 1
    # Binary search for the largest snap_id <= target snap_id
    while left <= right:
      mid = (left + right) // 2
      if history[mid][0] <= snap_id:
        left = mid + 1
      else:
        right = mid - 1
      result = history[right][1]
    print(f"Result found: {result} (Matches record: {history[right]})\n")
    return result

In [ ]:
sa = SnapshotArray(3)
sa.set(0, 5)
# print(self.arr)
sa.snap()
sa.set(0, 6)
sa.get(0, 0)

Array initialized with length 3. Current snap_id: 0
Current internal state: [[(0, 0)], [(0, 0)], [(0, 0)]]

index[0] = 5 (recorded during snap_id 0)
Current internal state: [[(0, 0), (0, 5)], [(0, 0)], [(0, 0)]]

Closing snap_id 0.
Next snap_id will be 1

index[0] = 6 (recorded during snap_id 1)
Current internal state: [[(0, 0), (0, 5), (1, 6)], [(0, 0)], [(0, 0)]]

Target: index 0 at snapshot 0
Timeline to search: [(0, 0), (0, 5), (1, 6)]
Result found: 5 (Matches record: (0, 5))



5

### **Course Schedule**

In [ ]:
from typing import List

class courseSchedule:
  def canFinish(self, numCourses: int, prerequisites: List[List[int]]) -> bool:
    preMap = {i:[] for i in range(numCourses)}
    for crs, pre in prerequisites:
      preMap[crs].append(pre)

    print("Prerequisites Hash Map")
    print("------------------------------")
    print(preMap)
    print("------------------------------")

    visitSet = set()
    def dfs(crs):
      if crs in visitSet:
        return False
      if preMap[crs] == []:
        return True

      visitSet.add(crs)
      for pre in preMap[crs]:
        if not dfs(pre):
          return False

      print(f"Visited {crs}")
      print("------")
      visitSet.remove(crs)
      preMap[crs] = []
      print(f"Updated preMap {preMap}")
      print("------------------------------------------------")
      return True

    for crs in range(numCourses):
      if not dfs(crs):
        print("The Course schedule is invalid")
        return False

    print("The Course schedule is valid")
    return True

In [ ]:
cs = courseSchedule()
print("#########################")
print("Example 1")
print("#########################")
cs.canFinish(5, [[0, 1], [0, 2], [1, 3], [1, 4], [3, 4]])
print(" ")
print("#########################")
print("Example 2")
print("#########################")
cs.canFinish(5, [[0, 1], [0, 2], [1, 3], [1, 4], [3, 1]])

#########################
Example 1
#########################
Prerequisites Hash Map
------------------------------
{0: [1, 2], 1: [3, 4], 2: [], 3: [4], 4: []}
------------------------------
Visited 3
------
Updated preMap {0: [1, 2], 1: [3, 4], 2: [], 3: [], 4: []}
------------------------------------------------
Visited 1
------
Updated preMap {0: [1, 2], 1: [], 2: [], 3: [], 4: []}
------------------------------------------------
Visited 0
------
Updated preMap {0: [], 1: [], 2: [], 3: [], 4: []}
------------------------------------------------
The Course schedule is valid
 
#########################
Example 2
#########################
Prerequisites Hash Map
------------------------------
{0: [1, 2], 1: [3, 4], 2: [], 3: [1], 4: []}
------------------------------
The Course schedule is invalid


False

### **Course Schedule 2**

In [ ]:
from typing import List

class courseSchedule2:
  def findOrder(self, numCourses: int, prerequisites: List[List[int]]) -> List[int]:
    preMap = {i:[] for i in range(numCourses)}
    for crs, pre in prerequisites:
      preMap[crs].append(pre)

    print("Prerequisites Initial Map")
    print("------------------------------")
    print(preMap)
    print("")


    res = []
    visit, cycle = set(), set()
    def dfs(crs):
      if crs in cycle:
        return False
      if crs in visit:
        return True

      cycle.add(crs)
      for pre in preMap[crs]:
        if dfs(pre) == False:
          return False

      print(f"Visited {crs}")
      print("------")
      cycle.remove(crs)
      visit.add(crs)
      res.append(crs)
      print(f"Updated preMap {preMap}")
      print("------------------------------------------------")
      print("")

    for crs in range(numCourses):
      if dfs(crs) == False:
        return []

    print(f"The Course schedule is valid: {res}")
    print("------------------------------------------------")
    print("")
    return res

In [ ]:
cs2 = courseSchedule2()
cs2.findOrder(6, [[0, 1], [0, 2], [1, 3], [3, 2], [4, 0], [5, 0]])
cs2.findOrder(6, [[0, 1], [0, 2], [1, 3], [2, 1], [3, 2], [4, 0], [5, 0]])

Prerequisites Initial Map
------------------------------
{0: [1, 2], 1: [3], 2: [], 3: [2], 4: [0], 5: [0]}

Visited 2
------
Updated preMap {0: [1, 2], 1: [3], 2: [], 3: [2], 4: [0], 5: [0]}
------------------------------------------------

Visited 3
------
Updated preMap {0: [1, 2], 1: [3], 2: [], 3: [2], 4: [0], 5: [0]}
------------------------------------------------

Visited 1
------
Updated preMap {0: [1, 2], 1: [3], 2: [], 3: [2], 4: [0], 5: [0]}
------------------------------------------------

Visited 0
------
Updated preMap {0: [1, 2], 1: [3], 2: [], 3: [2], 4: [0], 5: [0]}
------------------------------------------------

Visited 4
------
Updated preMap {0: [1, 2], 1: [3], 2: [], 3: [2], 4: [0], 5: [0]}
------------------------------------------------

Visited 5
------
Updated preMap {0: [1, 2], 1: [3], 2: [], 3: [2], 4: [0], 5: [0]}
------------------------------------------------

The Course schedule is valid: [2, 3, 1, 0, 4, 5]
--------------------------------------------

[]

### **Meeting Rooms**

In [ ]:
from typing import List

class meetingRooms:
  def canAttendMeetings(self, intervals: List[List[int]]) -> bool:
    print(f"Intervals: {intervals}")
    intervals.sort(key=lambda x:x[0])
    print(f"Sorted Intervals: {intervals}")
    for i in range(len(intervals) - 1):
      print(f"Iteration-{i + 1}")
      print(f"Interval-{i + 1}: {intervals[i][1]},  Interval-{i + 2}: {intervals[i + 1][0]}")
      if intervals[i][1] > intervals[i + 1][0]:
        return False
    return True

    # Time: O(nlogn)
    # Space: O(1)

In [ ]:
mr = meetingRooms()
mr.canAttendMeetings([[25, 30], [5, 10], [15, 20], [11, 14], [21, 24], [20, 40]])

Intervals: [[25, 30], [5, 10], [15, 20], [11, 14], [21, 24], [20, 40]]
Sorted Intervals: [[5, 10], [11, 14], [15, 20], [20, 40], [21, 24], [25, 30]]
Iteration-1
Interval-1: 10,  Interval-2: 11
Iteration-2
Interval-2: 14,  Interval-3: 15
Iteration-3
Interval-3: 20,  Interval-4: 20
Iteration-4
Interval-4: 40,  Interval-5: 21


False

### **Meeting Rooms II**

In [ ]:
import heapq

class meetingRooms2:
  def minMeetingRooms(self, intervals: List[List[int]]) -> int:
    print(f"Intervals: {intervals}")
    if not intervals:
      return 0

    # sort based on start times of the meetings
    intervals.sort(key=lambda x: x[0])
    print(f"Sorted Intervals: {intervals}")

    # initialize minHeap to store end times
    rooms = []
    print(f"rooms: {rooms}")

    # put the end of 1st meeting in minHeap
    heapq.heappush(rooms, intervals[0][1])
    print(f"rooms: {rooms}")

    for i in range(1, len(intervals)):
      # if currenbnt meeting starts after the lase earliest meeting ends
      print("")
      print(f"Iteration: {i}")
      print(f"Current Meeting: {intervals[i]}")
      print(f"Earliest Meeting: {rooms[0]}")
      if intervals[i][0] >= rooms[0]:
        # reuse the room
        heapq.heappop(rooms)

      # put the new end time into minHeap
      heapq.heappush(rooms, intervals[i][1])
      print(f"Updated rooms: {rooms}")
      print("")

    print(f"The minimum number of rooms required: {len(rooms)}")
    #return len(rooms)

    # Time: O(nlogn)
    # Spaca: O(n)

In [ ]:
mr2 = meetingRooms2()
mr2.minMeetingRooms([[25, 30], [5, 10], [15, 20], [11, 14], [21, 24], [20, 40]])

Intervals: [[25, 30], [5, 10], [15, 20], [11, 14], [21, 24], [20, 40]]
Sorted Intervals: [[5, 10], [11, 14], [15, 20], [20, 40], [21, 24], [25, 30]]
rooms: []
rooms: [10]

Iteration: 1
Current Meeting: [11, 14]
Earliest Meeting: 10
Updated rooms: [14]


Iteration: 2
Current Meeting: [15, 20]
Earliest Meeting: 14
Updated rooms: [20]


Iteration: 3
Current Meeting: [20, 40]
Earliest Meeting: 20
Updated rooms: [40]


Iteration: 4
Current Meeting: [21, 24]
Earliest Meeting: 40
Updated rooms: [24, 40]


Iteration: 5
Current Meeting: [25, 30]
Earliest Meeting: 24
Updated rooms: [30, 40]

The minimum number of rooms required: 2


### **Container with most water**

In [ ]:
class containerWithMostWater:
  def maxArea(self, height: List[int]) -> int:
    print(f"Height: {height}")
    max_area = 0
    l, r = 0, len(height) - 1
    print(f"l: {l}, r: {r}")

    while l < r:
      width = r - l
      print("")
      print(f"Width: {width}")
      print(f"Height {min(height[l], height[r])}")
      area = min(height[l], height[r]) * width
      print(f"Area: {area}")
      max_area = max(max_area, area)
      print(f"Max Area: {max_area}")

      if height[l] < height[r]:
        l += 1
      else:
        r -= 1

    return max_area

    # Time: O(n)
    # Space: O(1)

In [ ]:
cwm = containerWithMostWater()
cwm.maxArea([1,8,6,2,5,4,8,3,7])

Height: [1, 8, 6, 2, 5, 4, 8, 3, 7]
l: 0, r: 8

Width: 8
Height 1
Area: 8
Max Area: 8

Width: 7
Height 7
Area: 49
Max Area: 49

Width: 6
Height 3
Area: 18
Max Area: 49

Width: 5
Height 8
Area: 40
Max Area: 49

Width: 4
Height 4
Area: 16
Max Area: 49

Width: 3
Height 5
Area: 15
Max Area: 49

Width: 2
Height 2
Area: 4
Max Area: 49

Width: 1
Height 6
Area: 6
Max Area: 49


49

### **Find First and Last Position of Element in Sorted Array**

In [47]:
from typing import List

class Solution:
    def searchRange(self, nums: List[int], target: int) -> List[int]:
        left = self.binarySearch(nums, target, True)
        right = self.binarySearch(nums, target, False)
        return [left, right]

    def binarySearch(Self, nums, target, leftBias):
        l, r = 0, len(nums) - 1
        i = -1
        while l <= r:
            m = (l + r) // 2
            if target > nums[m]:
                l = m + 1
            elif target < nums[m]:
                r = m - 1
            else:
                i = m
                if leftBias:
                    r = m - 1
                else:
                    l = m + 1
        return i

In [48]:
def trace_binary_search(nums: List[int], target: int, leftBias: bool) -> int:
    """A verbose version to show each step of the search."""
    l, r = 0, len(nums) - 1
    i = -1
    step = 0

    print(f"\nTracing {'LEFT' if leftBias else 'RIGHT'}-biased binary search for target={target}")
    print(f"nums = {nums}")

    while l <= r:
        step += 1
        m = (l + r) // 2
        print(f"Step {step}: l={l}, r={r}, m={m}, nums[m]={nums[m]}, best_i={i}")

        if target > nums[m]:
            l = m + 1
            print("  target > nums[m] -> move l right")
        elif target < nums[m]:
            r = m - 1
            print("  target < nums[m] -> move r left")
        else:
            i = m
            print("  found target -> update best_i =", i)
            if leftBias:
                r = m - 1
                print("  leftBias -> move r left to keep searching earlier indices")
            else:
                l = m + 1
                print("  rightBias -> move l right to keep searching later indices")

    print("Done. Returning index:", i)
    return i


def run_tests():
    sol = Solution()

    test_cases = [
        # typical duplicates
        ([5, 7, 7, 8, 8, 10], 8, [3, 4]),
        ([5, 7, 7, 8, 8, 10], 7, [1, 2]),
        # target not present
        ([5, 7, 7, 8, 8, 10], 6, [-1, -1]),
        # all same
        ([2, 2, 2, 2], 2, [0, 3]),
        # single element
        ([1], 1, [0, 0]),
        ([1], 0, [-1, -1]),
        # empty list
        ([], 3, [-1, -1]),
        # target at edges
        ([1, 2, 3, 3, 3, 4], 1, [0, 0]),
        ([1, 2, 3, 3, 3, 4], 4, [5, 5]),
    ]

    print("=== Running Unit Tests ===")
    for nums, target, expected in test_cases:
        got = sol.searchRange(nums, target)
        status = "PASS" if got == expected else "FAIL"
        print(f"{status}: nums={nums}, target={target} -> got={got}, expected={expected}")

    # One detailed trace demo
    nums = [5, 7, 7, 8, 8, 10]
    target = 8
    print("\n=== Detailed Simulation Example ===")
    left = trace_binary_search(nums, target, leftBias=True)
    right = trace_binary_search(nums, target, leftBias=False)
    print("\nFinal range:", [left, right])


if __name__ == "__main__":
    run_tests()

=== Running Unit Tests ===
PASS: nums=[5, 7, 7, 8, 8, 10], target=8 -> got=[3, 4], expected=[3, 4]
PASS: nums=[5, 7, 7, 8, 8, 10], target=7 -> got=[1, 2], expected=[1, 2]
PASS: nums=[5, 7, 7, 8, 8, 10], target=6 -> got=[-1, -1], expected=[-1, -1]
PASS: nums=[2, 2, 2, 2], target=2 -> got=[0, 3], expected=[0, 3]
PASS: nums=[1], target=1 -> got=[0, 0], expected=[0, 0]
PASS: nums=[1], target=0 -> got=[-1, -1], expected=[-1, -1]
PASS: nums=[], target=3 -> got=[-1, -1], expected=[-1, -1]
PASS: nums=[1, 2, 3, 3, 3, 4], target=1 -> got=[0, 0], expected=[0, 0]
PASS: nums=[1, 2, 3, 3, 3, 4], target=4 -> got=[5, 5], expected=[5, 5]

=== Detailed Simulation Example ===

Tracing LEFT-biased binary search for target=8
nums = [5, 7, 7, 8, 8, 10]
Step 1: l=0, r=5, m=2, nums[m]=7, best_i=-1
  target > nums[m] -> move l right
Step 2: l=3, r=5, m=4, nums[m]=8, best_i=-1
  found target -> update best_i = 4
  leftBias -> move r left to keep searching earlier indices
Step 3: l=3, r=3, m=3, nums[m]=8, best_i